# RAG with OpenAI Native

Lab for implementing Retrieval-Augmented Generation using OpenAI's native capabilities.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("OpenAI key loaded:", OPENAI_API_KEY is not None)
print("Pinecone key loaded:", PINECONE_API_KEY is not None)

OpenAI key loaded: True
Pinecone key loaded: True


In [2]:
from openai import OpenAI
from PyPDF2 import PdfReader
import tiktoken
import numpy as np

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client ready")

OpenAI client ready


In [3]:
pdf_path = "data/Dissertation.pdf"

reader = PdfReader(pdf_path)

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print(f"Pages with extracted text: {len(pages)}")
print(pages[0]["text"][:500])

Pages with extracted text: 44
30021853  DISSERTATION  MAY 2009  
 
Page | 0  
 
“THE ILLUMINATI AND THE TOTALITARIAN TIP -TOE” 
 
SHOULD CONSPIRACY THEORIES BE COVERED  IN THE MAINSTREAM MEDIA?  
 
 
 
 (Student Number – 30021853 ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
May 2009  
 
Word  count: 9,070  
 
 
 
 
Submitted in partial fulfillment of  
the requirements for the degree of B.A.(Hons)  
 
 
Media Studies Degree awarded by Liverpool Hope University and studied at 
Wirral Metropolitan College  
 
 
 
 
 
 
 
 
 
 
 
 
 
  


In [4]:
total_characters = sum(len(page["text"]) for page in pages)

print(f"Pages extracted: {len(pages)}")
print(f"Total characters: {total_characters:,}")

Pages extracted: 44
Total characters: 76,305


In [5]:
chunk_size = 1000
overlap = 200

chunks = []

for page in pages:
    text = page["text"]

    for i in range(0, len(text), chunk_size - overlap):
        chunk_text = text[i:i + chunk_size]

        if chunk_text.strip():
            chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

print(f"Total chunks: {len(chunks)}")
print(chunks[0]["text"][:300])

Total chunks: 115
30021853  DISSERTATION  MAY 2009  
 
Page | 0  
 
“THE ILLUMINATI AND THE TOTALITARIAN TIP -TOE” 
 
SHOULD CONSPIRACY THEORIES BE COVERED  IN THE MAINSTREAM MEDIA?  
 
 
 
 (Student Number – 30021853 ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
May 2009  
 
Word  count: 9,070  
 
 
 
 
Submitted in partial fulf


In [7]:
print(chunks[10]["text"][:1000])

d to silence dissident 
government voices such as anti -war protesto rs.  
 
A documentary credited to the director Dylan Avery titled Loose Change became 
widely available over the internet and spread rapidly. The film gave an alternative 
version of the events of 9/11 and posed many interesting questions implying that the 
official version of the events of that fateful day was a lie and that it was an inside job.  
Loose Change  is one of many „investigative‟ documentaries on the subject of 9/11, 
the related concept of the New World Order, Illuminati and the insinuation of a long 
lost hidden knowledge that the majority of the world are kept from through the 
actions of this elite.  
 
A film claimed to riddled with so many inaccuracies and misinformation that it 
prompted one author to label it  
 
“perhaps the most factually challenged d ocumentary I‟ve ever seen...This 
is an amateurish piece of panic -peddling nonsense riddled with half -
truths, rehashed rumors, and implausible

In [8]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks[10]["text"]
)

embedding = response.data[0].embedding

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
1536
[0.01192474365234375, -0.0011453628540039062, -0.029388427734375, 0.0287017822265625, -0.0460205078125, -0.0190582275390625, -0.0211334228515625, 0.0241851806640625, -0.042510986328125, -0.029937744140625]


In [9]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

print(f"Embeddings added to {len(chunks)} chunks")
print(len(chunks[0]["embedding"]))

Embeddings added to 115 chunks
1536
